# Step 1: raw dataset generation

In [6]:
import pickle
import torch
import warnings
import sys
from pathlib import Path
from typing import List, Dict, Tuple, Any
from datasets import Dataset

def generate_raw_rhm_dataset(
    config_list: List[Tuple[int, int]], 
    samples_per_config: int = 1000,
    output_dir: str = './raw_rhm_data',
    vocab_size: int = 32,
    num_classes: int = 10,
    tuple_size: int = 2,
    seed_sample: int = 42,
    save_intermediate: bool = True
) -> Tuple[Dataset, Dict[str, Any]]:
    """
    Generate raw RHM dataset for multiple hierarchical configurations.
    
    Args:
        config_list: List of (L, m) tuples where L=hierarchy depth, m=multiplicity
        samples_per_config: Number of samples to generate per configuration
        output_dir: Directory to save the raw dataset
        vocab_size: Vocabulary size for RHM (default: 32)
        num_classes: Number of classes for RHM (default: 10)
        tuple_size: Size of low-level representations (default: 2)
        seed_sample: Seed for sample generation (default: 42)
        save_intermediate: Whether to save intermediate results per config
    
    Returns:
        dataset: HuggingFace Dataset containing all raw sequences
        metadata: Dictionary with comprehensive metadata
    """
    
    print("="*60)
    print("GENERATING RAW RHM DATASET")
    print("="*60)
    
    # Convert to Path object
    output_path = Path(output_dir)
    
    # Initialize storage for all sequences and metadata
    all_sequences = []
    all_task_ids = []
    all_config_L = []
    all_config_m = []
    all_sequence_lengths = []
    all_rules = {}
    
    # Statistics tracking
    total_sequences = 0
    config_stats = {}
    
    print(f"Configurations to generate: {len(config_list)}")
    print(f"Samples per configuration: {samples_per_config}")
    print(f"Total target sequences: {len(config_list) * samples_per_config}")
    print()
    
    # Generate data for each configuration
    for task_id, (L, m) in enumerate(config_list):
        print(f"Generating Task {task_id}: L={L} (depth), m={m} (multiplicity)")
        print("-" * 50)
        
        try:
            # Create RHM instance for this configuration
            rhm = RandomHierarchyModel(
                num_features=vocab_size,      # vocabulary size (0-31, +1 shift makes it 1-32)
                num_classes=num_classes,      # number of classes
                num_synonyms=m,               # multiplicity parameter
                tuple_size=tuple_size,        # size of low-level representations
                num_layers=L,                 # hierarchy depth
                seed_rules=task_id,           # unique rules per task
                seed_sample=seed_sample,      # consistent sampling across tasks
                train_size=samples_per_config,
                test_size=0,
                input_format='long',          # integer sequences (1-based indexing)
                replacement=True              # allow sampling with replacement
            )
            
            # Extract generated data
            sequences = rhm.features      # Shape: [samples_per_config, variable_length]
            labels = rhm.labels          # Shape: [samples_per_config] (not used in LM)
            rules = rhm.rules            # Production rules dictionary
            
            # Convert tensors to lists for HuggingFace compatibility
            if hasattr(sequences, 'tolist'):
                sequences_list = sequences.tolist()
            else:
                sequences_list = [list(seq) for seq in sequences]
            
            # Calculate statistics for this configuration
            seq_lengths = [len(seq) for seq in sequences_list]
            config_stats[task_id] = {
                'L': L,
                'm': m,
                'num_sequences': len(sequences_list),
                'min_length': min(seq_lengths),
                'max_length': max(seq_lengths),
                'avg_length': sum(seq_lengths) / len(seq_lengths),
                'total_tokens': sum(seq_lengths)
            }
            
            # Store rules for this configuration
            all_rules[task_id] = {
                'L': L,
                'm': m,
                'rules_dict': rules,
                'vocab_range': f"1-{vocab_size}",  # RHM uses 1-based indexing
                'num_sequences': len(sequences_list)
            }
            
            # Add to master dataset
            all_sequences.extend(sequences_list)
            all_task_ids.extend([task_id] * len(sequences_list))
            all_config_L.extend([L] * len(sequences_list))
            all_config_m.extend([m] * len(sequences_list))
            all_sequence_lengths.extend(seq_lengths)
            
            total_sequences += len(sequences_list)
            
            # Print statistics for this configuration
            print(f"  ✓ Generated {len(sequences_list)} sequences")
            print(f"  ✓ Length range: {min(seq_lengths)}-{max(seq_lengths)} tokens")
            print(f"  ✓ Average length: {sum(seq_lengths)/len(seq_lengths):.1f} tokens")
            print(f"  ✓ Total tokens: {sum(seq_lengths):,}")
            
            # Save intermediate results if requested
            if save_intermediate:
                config_dir = output_path / 'intermediate' / f'task_{task_id}_L{L}_m{m}'
                config_dir.mkdir(parents=True, exist_ok=True)
                
                # Save sequences and metadata for this config
                config_data = {
                    'sequences': sequences_list,
                    'task_id': task_id,
                    'L': L,
                    'm': m,
                    'rules': rules,
                    'stats': config_stats[task_id]
                }
                
                with (config_dir / 'config_data.pkl').open('wb') as f:
                    pickle.dump(config_data, f)
                
                print(f"  ✓ Saved intermediate results to {config_dir}")
            
            print()
            
        except Exception as e:
            print(f"  ✗ Error generating task {task_id} (L={L}, m={m}): {e}")
            print(f"  ✗ Skipping this configuration...")
            print()
            continue
    
    # Create comprehensive metadata
    metadata = {
        'generation_params': {
            'vocab_size': vocab_size,
            'num_classes': num_classes,
            'tuple_size': tuple_size,
            'seed_sample': seed_sample,
            'samples_per_config': samples_per_config
        },
        'configurations': [
            {'task_id': i, 'L': L, 'm': m} 
            for i, (L, m) in enumerate(config_list)
        ],
        'config_stats': config_stats,
        'rules': all_rules,
        'dataset_stats': {
            'total_sequences': total_sequences,
            'total_configs': len(config_list),
            'successful_configs': len(config_stats),
            'min_seq_length': min(all_sequence_lengths) if all_sequence_lengths else 0,
            'max_seq_length': max(all_sequence_lengths) if all_sequence_lengths else 0,
            'avg_seq_length': sum(all_sequence_lengths) / len(all_sequence_lengths) if all_sequence_lengths else 0,
            'total_tokens': sum(all_sequence_lengths),
            'vocab_range': f"1-{vocab_size} (0 reserved for special tokens)"
        }
    }
    
    # Create HuggingFace Dataset
    print("Creating HuggingFace Dataset...")
    dataset_dict = {
        'input_ids': all_sequences,           # Raw integer sequences
        'task_id': all_task_ids,              # Which configuration generated this sequence
        'config_L': all_config_L,             # Hierarchy depth for this sequence
        'config_m': all_config_m,             # Multiplicity for this sequence
        'length': all_sequence_lengths        # Length of this sequence
    }
    
    dataset = Dataset.from_dict(dataset_dict)
    
    # Save complete dataset and metadata
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Save HuggingFace dataset
    dataset.save_to_disk(str(output_path / 'raw_dataset'))
    print(f"✓ Saved HuggingFace dataset to {output_path / 'raw_dataset'}")
    
    # Save metadata
    with (output_path / 'metadata.pkl').open('wb') as f:
        pickle.dump(metadata, f)
    print(f"✓ Saved metadata to {output_path / 'metadata.pkl'}")
    
    # Save human-readable summary
    with (output_path / 'dataset_summary.txt').open('w') as f:
        f.write("RHM DATASET GENERATION SUMMARY\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Total sequences: {total_sequences:,}\n")
        f.write(f"Total tokens: {sum(all_sequence_lengths):,}\n")
        f.write(f"Vocabulary: 1-{vocab_size} (0 reserved)\n")
        f.write(f"Sequence length range: {min(all_sequence_lengths)}-{max(all_sequence_lengths)}\n")
        f.write(f"Average sequence length: {sum(all_sequence_lengths)/len(all_sequence_lengths):.1f}\n\n")
        
        f.write("CONFIGURATION DETAILS:\n")
        f.write("-" * 30 + "\n")
        for task_id, stats in config_stats.items():
            f.write(f"Task {task_id}: L={stats['L']}, m={stats['m']}\n")
            f.write(f"  Sequences: {stats['num_sequences']:,}\n")
            f.write(f"  Length: {stats['min_length']}-{stats['max_length']} (avg: {stats['avg_length']:.1f})\n")
            f.write(f"  Tokens: {stats['total_tokens']:,}\n\n")
    
    print(f"✓ Saved summary to {output_path / 'dataset_summary.txt'}")
    
    # Final summary
    print("\n" + "="*60)
    print("RAW DATASET GENERATION COMPLETE")
    print("="*60)
    print(f"Total sequences generated: {total_sequences:,}")
    print(f"Total tokens: {sum(all_sequence_lengths):,}")
    print(f"Successful configurations: {len(config_stats)}/{len(config_list)}")
    print(f"Dataset saved to: {output_path}")
    print("="*60)
    
    return dataset, metadata



In [7]:

# Example usage function
def generate_example_dataset():
    """Generate an example RHM dataset with multiple configurations"""
    
    # Define hierarchical configurations to test
    config_list = [
        (2, 2),   # Shallow, low multiplicity
        (3, 2),   # Medium depth, low multiplicity  
        (2, 4),   # Shallow, high multiplicity
        (4, 2),   # Deep, low multiplicity
        (3, 3),   # Medium depth, medium multiplicity
    ]
    
    # Generate the dataset
    dataset, metadata = generate_raw_rhm_dataset(
        config_list=config_list,
        samples_per_config=1000,
        output_dir='./raw_rhm_data',
        vocab_size=32,
        num_classes=10,
        save_intermediate=True
    )
    
    return dataset, metadata


if __name__ == "__main__":
    # Generate example dataset
    dataset, metadata = generate_example_dataset()
    
    # Quick inspection
    print(f"\nDataset inspection:")
    print(f"Number of sequences: {len(dataset)}")
    print(f"First sequence: {dataset[0]['input_ids'][:20]}...")  # Show first 20 tokens
    print(f"First sequence length: {dataset[0]['length']}")
    print(f"First sequence config: L={dataset[0]['config_L']}, m={dataset[0]['config_m']}")

GENERATING RAW RHM DATASET
Configurations to generate: 5
Samples per configuration: 1000
Total target sequences: 5000

Generating Task 0: L=2 (depth), m=2 (multiplicity)
--------------------------------------------------
  ✓ Generated 1000 sequences
  ✓ Length range: 4-4 tokens
  ✓ Average length: 4.0 tokens
  ✓ Total tokens: 4,000
  ✓ Saved intermediate results to raw_rhm_data/intermediate/task_0_L2_m2

Generating Task 1: L=3 (depth), m=2 (multiplicity)
--------------------------------------------------
  ✓ Generated 1000 sequences
  ✓ Length range: 8-8 tokens
  ✓ Average length: 8.0 tokens
  ✓ Total tokens: 8,000
  ✓ Saved intermediate results to raw_rhm_data/intermediate/task_1_L3_m2

Generating Task 2: L=2 (depth), m=4 (multiplicity)
--------------------------------------------------
  ✓ Generated 1000 sequences
  ✓ Length range: 4-4 tokens
  ✓ Average length: 4.0 tokens
  ✓ Total tokens: 4,000
  ✓ Saved intermediate results to raw_rhm_data/intermediate/task_2_L2_m4

Generating Tas

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

✓ Saved HuggingFace dataset to raw_rhm_data/raw_dataset
✓ Saved metadata to raw_rhm_data/metadata.pkl
✓ Saved summary to raw_rhm_data/dataset_summary.txt

RAW DATASET GENERATION COMPLETE
Total sequences generated: 5,000
Total tokens: 40,000
Successful configurations: 5/5
Dataset saved to: raw_rhm_data

Dataset inspection:
Number of sequences: 5000
First sequence: [10, 19, 32, 17]...
First sequence length: 4
First sequence config: L=2, m=2


# Step 2: Unified HF dataset

In [2]:
import pickle
import json
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple
from datasets import Dataset, load_from_disk
import torch

class UnifiedRHMDataset:
    """
    Unified interface for RHM datasets that can be used across different training paradigms.
    Loads raw RHM data and provides task-agnostic access with rich metadata.
    """
    
    def __init__(self, dataset_path: str):
        """
        Initialize unified dataset from saved raw RHM data.
        
        Args:
            dataset_path: Path to directory containing raw_dataset and metadata.pkl
        """
        self.dataset_path = Path(dataset_path)
        self.dataset = None
        self.metadata = None
        
        self._load_dataset()
        self._validate_dataset()
        self._compute_statistics()
    
    def _load_dataset(self):
        """Load the HuggingFace dataset and metadata"""
        print("Loading RHM dataset...")
        
        # Load HuggingFace dataset
        dataset_file = self.dataset_path / 'raw_dataset'
        if not dataset_file.exists():
            raise FileNotFoundError(f"Dataset not found at {dataset_file}")
        
        self.dataset = load_from_disk(str(dataset_file))
        print(f"✓ Loaded dataset with {len(self.dataset)} sequences")
        
        # Load metadata
        metadata_file = self.dataset_path / 'metadata.pkl'
        if not metadata_file.exists():
            raise FileNotFoundError(f"Metadata not found at {metadata_file}")
        
        with metadata_file.open('rb') as f:
            self.metadata = pickle.load(f)
        print(f"✓ Loaded metadata for {len(self.metadata['configurations'])} configurations")
    
    def _validate_dataset(self):
        """Validate dataset structure and consistency"""
        print("Validating dataset structure...")
        
        # Check required columns
        required_columns = ['input_ids', 'task_id', 'config_L', 'config_m', 'length']
        missing_columns = [col for col in required_columns if col not in self.dataset.column_names]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")
        
        # Check data consistency
        assert len(self.dataset) > 0, "Dataset is empty"
        
        # Validate sequence lengths match recorded lengths
        sample_indices = range(min(100, len(self.dataset)))  # Check first 100 samples
        for i in sample_indices:
            recorded_length = self.dataset[i]['length']
            actual_length = len(self.dataset[i]['input_ids'])
            assert recorded_length == actual_length, f"Length mismatch at index {i}: recorded={recorded_length}, actual={actual_length}"
        
        # Check task_id consistency
        unique_task_ids = set(self.dataset['task_id'])
        metadata_task_ids = set(config['task_id'] for config in self.metadata['configurations'])
        assert unique_task_ids == metadata_task_ids, f"Task ID mismatch: dataset={unique_task_ids}, metadata={metadata_task_ids}"
        
        print("✓ Dataset validation passed")
    
    def _compute_statistics(self):
        """Compute and cache dataset statistics"""
        print("Computing dataset statistics...")
        
        # Overall statistics
        lengths = self.dataset['length']
        self.stats = {
            'total_sequences': len(self.dataset),
            'total_tokens': sum(lengths),
            'min_length': min(lengths),
            'max_length': max(lengths),
            'avg_length': sum(lengths) / len(lengths),
            'vocab_size': self.metadata['generation_params']['vocab_size'],
            'vocab_range': f"1-{self.metadata['generation_params']['vocab_size']} (0 reserved)"
        }
        
        # Per-configuration statistics
        self.config_stats = {}
        for config in self.metadata['configurations']:
            task_id = config['task_id']
            L = config['L']
            m = config['m']
            
            # Filter sequences for this configuration
            config_mask = [tid == task_id for tid in self.dataset['task_id']]
            config_lengths = [length for length, mask in zip(lengths, config_mask) if mask]
            config_sequences = sum(config_mask)
            
            self.config_stats[task_id] = {
                'L': L,
                'm': m,
                'num_sequences': config_sequences,
                'min_length': min(config_lengths) if config_lengths else 0,
                'max_length': max(config_lengths) if config_lengths else 0,
                'avg_length': sum(config_lengths) / len(config_lengths) if config_lengths else 0,
                'total_tokens': sum(config_lengths),
                'proportion': config_sequences / len(self.dataset)
            }
        
        print("✓ Statistics computed")
    
    def get_dataset(self) -> Dataset:
        """Get the raw HuggingFace dataset"""
        return self.dataset
    
    def get_metadata(self) -> Dict[str, Any]:
        """Get complete metadata"""
        return self.metadata
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get computed statistics"""
        return {
            'overall': self.stats,
            'per_config': self.config_stats
        }
    
    def get_vocab_info(self) -> Dict[str, Any]:
        """Get vocabulary information"""
        return {
            'vocab_size': self.stats['vocab_size'],
            'vocab_range': self.stats['vocab_range'],
            'reserved_tokens': {
                0: 'PAD/EOS/SEP',
                'vocab_size + 1': 'MASK (for MLM)',
                'vocab_size + 2': 'CLS (if needed)', 
                'vocab_size + 3': 'BOS (if needed)'
            },
            'effective_vocab_size': self.stats['vocab_size'] + 4  # Including special tokens
        }
    
    def filter_by_config(self, L: Optional[int] = None, m: Optional[int] = None, 
                        task_ids: Optional[List[int]] = None) -> Dataset:
        """
        Filter dataset by hierarchical configuration parameters.
        
        Args:
            L: Hierarchy depth to filter by
            m: Multiplicity to filter by  
            task_ids: Specific task IDs to include
            
        Returns:
            Filtered HuggingFace dataset
        """
        indices_to_keep = []
        
        for i in range(len(self.dataset)):
            keep = True
            
            if L is not None and self.dataset[i]['config_L'] != L:
                keep = False
            if m is not None and self.dataset[i]['config_m'] != m:
                keep = False
            if task_ids is not None and self.dataset[i]['task_id'] not in task_ids:
                keep = False
                
            if keep:
                indices_to_keep.append(i)
        
        return self.dataset.select(indices_to_keep)
    
    def filter_by_length(self, min_length: Optional[int] = None, 
                        max_length: Optional[int] = None) -> Dataset:
        """
        Filter dataset by sequence length.
        
        Args:
            min_length: Minimum sequence length (inclusive)
            max_length: Maximum sequence length (inclusive)
            
        Returns:
            Filtered HuggingFace dataset
        """
        indices_to_keep = []
        
        for i in range(len(self.dataset)):
            length = self.dataset[i]['length']
            keep = True
            
            if min_length is not None and length < min_length:
                keep = False
            if max_length is not None and length > max_length:
                keep = False
                
            if keep:
                indices_to_keep.append(i)
        
        return self.dataset.select(indices_to_keep)
    
    def get_config_groups(self) -> Dict[Tuple[int, int], List[int]]:
        """
        Group sequence indices by (L, m) configuration.
        
        Returns:
            Dictionary mapping (L, m) tuples to lists of sequence indices
        """
        config_groups = {}
        
        for i in range(len(self.dataset)):
            L = self.dataset[i]['config_L']
            m = self.dataset[i]['config_m']
            config_key = (L, m)
            
            if config_key not in config_groups:
                config_groups[config_key] = []
            config_groups[config_key].append(i)
        
        return config_groups
    
    def get_length_groups(self, bucket_size: int = 50) -> Dict[str, List[int]]:
        """
        Group sequence indices by length buckets.
        
        Args:
            bucket_size: Size of each length bucket
            
        Returns:
            Dictionary mapping bucket names to lists of sequence indices
        """
        length_groups = {}
        
        for i in range(len(self.dataset)):
            length = self.dataset[i]['length']
            bucket_start = (length // bucket_size) * bucket_size
            bucket_end = bucket_start + bucket_size - 1
            bucket_name = f"{bucket_start}-{bucket_end}"
            
            if bucket_name not in length_groups:
                length_groups[bucket_name] = []
            length_groups[bucket_name].append(i)
        
        return length_groups
    
    def print_summary(self):
        """Print a comprehensive summary of the dataset"""
        print("\n" + "="*60)
        print("UNIFIED RHM DATASET SUMMARY")
        print("="*60)
        
        print(f"Total sequences: {self.stats['total_sequences']:,}")
        print(f"Total tokens: {self.stats['total_tokens']:,}")
        print(f"Sequence length: {self.stats['min_length']}-{self.stats['max_length']} (avg: {self.stats['avg_length']:.1f})")
        print(f"Vocabulary: {self.stats['vocab_range']}")
        
        print(f"\nConfigurations ({len(self.config_stats)}):")
        print("-" * 40)
        for task_id, stats in self.config_stats.items():
            print(f"Task {task_id}: L={stats['L']}, m={stats['m']}")
            print(f"  Sequences: {stats['num_sequences']:,} ({stats['proportion']:.1%})")
            print(f"  Length: {stats['min_length']}-{stats['max_length']} (avg: {stats['avg_length']:.1f})")
            print(f"  Tokens: {stats['total_tokens']:,}")
        
        print("="*60)
    
    def save_analysis(self, output_path: str):
        """Save dataset analysis to file"""
        output_file = Path(output_path)
        
        analysis = {
            'statistics': self.get_statistics(),
            'vocab_info': self.get_vocab_info(),
            'config_groups': {str(k): v for k, v in self.get_config_groups().items()},
            'sample_sequences': {
                'first_10_tokens': [seq[:10] for seq in self.dataset['input_ids'][:5]],
                'sequence_lengths': self.dataset['length'][:10]
            }
        }
        
        # Ensure parent directory exists
        output_file.parent.mkdir(parents=True, exist_ok=True)
        
        with output_file.open('w') as f:
            json.dump(analysis, f, indent=2)
        
        print(f"✓ Analysis saved to {output_file}")


/Users/jliu/anaconda3/envs/genai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


def load_unified_dataset(dataset_path: str) -> UnifiedRHMDataset:
    """
    Convenience function to load a unified RHM dataset.
    
    Args:
        dataset_path: Path to directory containing raw RHM dataset
        
    Returns:
        UnifiedRHMDataset instance
    """
    return UnifiedRHMDataset(dataset_path)


# Example usage and validation functions
def validate_dataset_loading(dataset_path: str):
    """Validate that dataset can be loaded and accessed properly"""
    print("Validating dataset loading...")
    
    # Load dataset
    unified_dataset = load_unified_dataset(dataset_path)
    
    # Test basic access
    dataset = unified_dataset.get_dataset()
    metadata = unified_dataset.get_metadata()
    stats = unified_dataset.get_statistics()
    vocab_info = unified_dataset.get_vocab_info()
    
    # Test filtering
    print("Testing filtering capabilities...")
    
    # Filter by configuration
    config_filtered = unified_dataset.filter_by_config(L=2, m=2)
    print(f"  L=2, m=2 filter: {len(config_filtered)} sequences")
    
    # Filter by length
    length_filtered = unified_dataset.filter_by_length(min_length=100, max_length=500)
    print(f"  Length 100-500 filter: {len(length_filtered)} sequences")
    
    # Test grouping
    config_groups = unified_dataset.get_config_groups()
    length_groups = unified_dataset.get_length_groups(bucket_size=100)
    
    print(f"  Configuration groups: {len(config_groups)}")
    print(f"  Length groups: {len(length_groups)}")
    
    # Print summary
    unified_dataset.print_summary()
    
    print("\n✓ Dataset validation completed successfully")
    
    return unified_dataset


if __name__ == "__main__":
    # Example: Load and validate a dataset
    dataset_path = './raw_rhm_data'  # Adjust path as needed
    
    try:
        unified_dataset = validate_dataset_loading(dataset_path)
        
        # Save analysis
        unified_dataset.save_analysis('./dataset_analysis.json')
        
    except FileNotFoundError as e:
        print(f"Dataset not found: {e}")
        print("Please run the raw dataset generation step first.")
    except Exception as e:
        print(f"Error loading dataset: {e}")

Validating dataset loading...
Loading RHM dataset...
✓ Loaded dataset with 5000 sequences
✓ Loaded metadata for 5 configurations
Validating dataset structure...
✓ Dataset validation passed
Computing dataset statistics...
✓ Statistics computed
Testing filtering capabilities...
  L=2, m=2 filter: 1000 sequences
  Length 100-500 filter: 0 sequences
  Configuration groups: 5
  Length groups: 1

UNIFIED RHM DATASET SUMMARY
Total sequences: 5,000
Total tokens: 40,000
Sequence length: 4-16 (avg: 8.0)
Vocabulary: 1-32 (0 reserved)

Configurations (5):
----------------------------------------
Task 0: L=2, m=2
  Sequences: 1,000 (20.0%)
  Length: 4-4 (avg: 4.0)
  Tokens: 4,000
Task 1: L=3, m=2
  Sequences: 1,000 (20.0%)
  Length: 8-8 (avg: 8.0)
  Tokens: 8,000
Task 2: L=2, m=4
  Sequences: 1,000 (20.0%)
  Length: 4-4 (avg: 4.0)
  Tokens: 4,000
Task 3: L=4, m=2
  Sequences: 1,000 (20.0%)
  Length: 16-16 (avg: 16.0)
  Tokens: 16,000
Task 4: L=3, m=3
  Sequences: 1,000 (20.0%)
  Length: 8-8 (avg: 8

# Step 3: Task-specific dataset

## Task-Specific Processing Layer

In [4]:
import torch
import random
from abc import ABC, abstractmethod
from typing import Dict, List, Any, Optional, Tuple
from datasets import Dataset
from dataclasses import dataclass

@dataclass
class TaskConfig:
    """Configuration for different language modeling tasks"""
    task_name: str
    max_length: Optional[int] = None
    pad_token_id: int = 0
    mask_token_id: int = 33  # vocab_size + 1
    cls_token_id: int = 34   # vocab_size + 2
    sep_token_id: int = 35   # vocab_size + 3
    
    # Task-specific parameters
    mask_probability: float = 0.15
    mask_strategy: str = 'random'  # 'random', 'hierarchical', 'level_specific'
    causal_mask: bool = True
    
    def __post_init__(self):
        if self.max_length is None:
            self.max_length = 512 if self.task_name == 'mlm' else 2048


class BaseTaskProcessor(ABC):
    """Base class for task-specific processors"""
    
    def __init__(self, config: TaskConfig, vocab_size: int = 32):
        self.config = config
        self.vocab_size = vocab_size
        self.effective_vocab_size = vocab_size + 4  # Including special tokens
    
    @abstractmethod
    def process_dataset(self, dataset: Dataset) -> Dataset:
        """Process dataset for the specific task"""
        pass
    
    @abstractmethod
    def get_collate_fn(self):
        """Get collate function for DataLoader"""
        pass
    
    def _validate_input(self, dataset: Dataset):
        """Validate input dataset has required columns"""
        required_columns = ['input_ids', 'task_id', 'config_L', 'config_m', 'length']
        missing = [col for col in required_columns if col not in dataset.column_names]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")


class CLMProcessor(BaseTaskProcessor):
    """Processor for Causal Language Modeling"""
    
    def __init__(self, config: TaskConfig, vocab_size: int = 32):
        super().__init__(config, vocab_size)
        self.config.causal_mask = True
    
    def process_dataset(self, dataset: Dataset) -> Dataset:
        """
        Process dataset for causal language modeling.
        Creates input_ids and labels with shifting.
        """
        self._validate_input(dataset)
        
        print(f"Processing dataset for Causal Language Modeling...")
        print(f"Original sequences: {len(dataset)}")
        
        processed_data = {
            'input_ids': [],
            'labels': [],
            'attention_mask': [],
            'task_id': [],
            'config_L': [],
            'config_m': [],
            'length': []
        }
        
        for example in dataset:
            sequence = example['input_ids']
            
            # Skip sequences that are too short
            if len(sequence) < 2:
                continue
            
            # Truncate if too long
            if self.config.max_length and len(sequence) > self.config.max_length:
                sequence = sequence[:self.config.max_length]
            
            # Create input and labels (shifted by 1)
            input_ids = sequence[:-1]  # All but last token
            labels = sequence[1:]      # All but first token
            
            # Create attention mask (all 1s for now, padding handled in collate_fn)
            attention_mask = [1] * len(input_ids)
            
            processed_data['input_ids'].append(input_ids)
            processed_data['labels'].append(labels)
            processed_data['attention_mask'].append(attention_mask)
            processed_data['task_id'].append(example['task_id'])
            processed_data['config_L'].append(example['config_L'])
            processed_data['config_m'].append(example['config_m'])
            processed_data['length'].append(len(input_ids))
        
        print(f"Processed sequences: {len(processed_data['input_ids'])}")
        print(f"Average length: {sum(processed_data['length']) / len(processed_data['length']):.1f}")
        
        return Dataset.from_dict(processed_data)
    
    def get_collate_fn(self):
        """Get collate function for CLM"""
        def collate_fn(batch):
            # Extract data
            input_ids = [torch.tensor(item['input_ids']) for item in batch]
            labels = [torch.tensor(item['labels']) for item in batch]
            
            # Pad sequences
            from torch.nn.utils.rnn import pad_sequence
            input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=self.config.pad_token_id)
            labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)  # -100 ignored in loss
            
            # Create attention mask
            attention_mask = (input_ids_padded != self.config.pad_token_id).long()
            
            return {
                'input_ids': input_ids_padded,
                'attention_mask': attention_mask,
                'labels': labels_padded,
                'task_ids': [item['task_id'] for item in batch],
                'config_L': [item['config_L'] for item in batch],
                'config_m': [item['config_m'] for item in batch]
            }
        
        return collate_fn


class MLMProcessor(BaseTaskProcessor):
    """Processor for Masked Language Modeling"""
    
    def __init__(self, config: TaskConfig, vocab_size: int = 32):
        super().__init__(config, vocab_size)
        self.config.causal_mask = False
    
    def process_dataset(self, dataset: Dataset) -> Dataset:
        """
        Process dataset for masked language modeling.
        Applies masking strategy to create input_ids and labels.
        """
        self._validate_input(dataset)
        
        print(f"Processing dataset for Masked Language Modeling...")
        print(f"Masking strategy: {self.config.mask_strategy}")
        print(f"Mask probability: {self.config.mask_probability}")
        print(f"Original sequences: {len(dataset)}")
        
        processed_data = {
            'input_ids': [],
            'labels': [],
            'attention_mask': [],
            'task_id': [],
            'config_L': [],
            'config_m': [],
            'length': []
        }
        
        for example in dataset:
            sequence = example['input_ids']
            
            # Skip sequences that are too short
            if len(sequence) < 2:
                continue
            
            # Truncate if too long
            if self.config.max_length and len(sequence) > self.config.max_length:
                sequence = sequence[:self.config.max_length]
            
            # Apply masking
            input_ids, labels = self._apply_masking(
                sequence, 
                example['config_L'], 
                example['config_m']
            )
            
            # Create attention mask
            attention_mask = [1] * len(input_ids)
            
            processed_data['input_ids'].append(input_ids)
            processed_data['labels'].append(labels)
            processed_data['attention_mask'].append(attention_mask)
            processed_data['task_id'].append(example['task_id'])
            processed_data['config_L'].append(example['config_L'])
            processed_data['config_m'].append(example['config_m'])
            processed_data['length'].append(len(input_ids))
        
        print(f"Processed sequences: {len(processed_data['input_ids'])}")
        print(f"Average length: {sum(processed_data['length']) / len(processed_data['length']):.1f}")
        
        return Dataset.from_dict(processed_data)
    
    def _apply_masking(self, sequence: List[int], config_L: int, config_m: int) -> Tuple[List[int], List[int]]:
        """Apply masking strategy to sequence"""
        
        if self.config.mask_strategy == 'random':
            return self._random_masking(sequence)
        elif self.config.mask_strategy == 'hierarchical':
            return self._hierarchical_masking(sequence, config_L, config_m)
        elif self.config.mask_strategy == 'level_specific':
            return self._level_specific_masking(sequence, config_L, config_m)
        else:
            raise ValueError(f"Unknown masking strategy: {self.config.mask_strategy}")
    
    def _random_masking(self, sequence: List[int]) -> Tuple[List[int], List[int]]:
        """Standard BERT-style random masking"""
        input_ids = sequence.copy()
        labels = [-100] * len(sequence)  # -100 = ignore in loss
        
        for i in range(len(sequence)):
            if random.random() < self.config.mask_probability:
                labels[i] = sequence[i]  # Store original token
                
                # 80% mask, 10% random, 10% keep
                rand = random.random()
                if rand < 0.8:
                    input_ids[i] = self.config.mask_token_id
                elif rand < 0.9:
                    input_ids[i] = random.randint(1, self.vocab_size)
                # else: keep original
        
        return input_ids, labels
    
    def _hierarchical_masking(self, sequence: List[int], config_L: int, config_m: int) -> Tuple[List[int], List[int]]:
        """
        Hierarchical-aware masking that respects the underlying structure.
        This is a simplified version - in practice, you'd use the actual rules.
        """
        # For now, implement as block masking to simulate hierarchical structure
        input_ids = sequence.copy()
        labels = [-100] * len(sequence)
        
        # Calculate approximate block size based on hierarchy
        block_size = max(2, len(sequence) // (config_L * config_m))
        
        i = 0
        while i < len(sequence):
            if random.random() < self.config.mask_probability:
                # Mask entire block
                block_end = min(i + block_size, len(sequence))
                for j in range(i, block_end):
                    labels[j] = sequence[j]
                    input_ids[j] = self.config.mask_token_id
                i = block_end
            else:
                i += 1
        
        return input_ids, labels
    
    def _level_specific_masking(self, sequence: List[int], config_L: int, config_m: int) -> Tuple[List[int], List[int]]:
        """
        Focus masking on specific positions that correspond to hierarchy levels.
        This is a simplified version.
        """
        input_ids = sequence.copy()
        labels = [-100] * len(sequence)
        
        # Focus on positions that are multiples of tuple_size (typically 2)
        tuple_size = 2  # This should come from metadata
        step = tuple_size ** config_L
        
        for i in range(0, len(sequence), step):
            if random.random() < self.config.mask_probability:
                labels[i] = sequence[i]
                input_ids[i] = self.config.mask_token_id
        
        return input_ids, labels
    
    def get_collate_fn(self):
        """Get collate function for MLM"""
        def collate_fn(batch):
            # Extract data
            input_ids = [torch.tensor(item['input_ids']) for item in batch]
            labels = [torch.tensor(item['labels']) for item in batch]
            
            # Pad sequences
            from torch.nn.utils.rnn import pad_sequence
            input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=self.config.pad_token_id)
            labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)
            
            # Create attention mask (bidirectional for MLM)
            attention_mask = (input_ids_padded != self.config.pad_token_id).long()
            
            return {
                'input_ids': input_ids_padded,
                'attention_mask': attention_mask,
                'labels': labels_padded,
                'task_ids': [item['task_id'] for item in batch],
                'config_L': [item['config_L'] for item in batch],
                'config_m': [item['config_m'] for item in batch]
            }
        
        return collate_fn


class ProcessorFactory:
    """Factory for creating task processors"""
    
    @staticmethod
    def create_processor(task_name: str, config: TaskConfig, vocab_size: int = 32) -> BaseTaskProcessor:
        """Create processor for specified task"""
        
        if task_name.lower() == 'clm':
            return CLMProcessor(config, vocab_size)
        elif task_name.lower() == 'mlm':
            return MLMProcessor(config, vocab_size)
        else:
            raise ValueError(f"Unknown task: {task_name}. Supported: ['clm', 'mlm']")

## Dynamic batching

In [5]:
# dynamic batching

import torch
from typing import Dict, List, Tuple, Optional, Any
from datasets import Dataset
from collections import defaultdict
import numpy as np

class DynamicBatcher:
    """
    Handles dynamic batching strategies for RHM datasets.
    Supports configuration-aware and length-aware batching.
    """
    
    def __init__(self, 
                 strategy: str = 'config_then_length',  # 'config_only', 'length_only', 'config_then_length'
                 length_bucket_size: int = 50,
                 max_batch_size: int = 32,
                 shuffle_within_groups: bool = True):
        """
        Initialize dynamic batcher.
        
        Args:
            strategy: Batching strategy to use
            length_bucket_size: Size of length buckets
            max_batch_size: Maximum batch size
            shuffle_within_groups: Whether to shuffle within groups
        """
        self.strategy = strategy
        self.length_bucket_size = length_bucket_size
        self.max_batch_size = max_batch_size
        self.shuffle_within_groups = shuffle_within_groups
    
    def create_batches(self, dataset: Dataset, seed: Optional[int] = None) -> List[List[int]]:
        """
        Create batches according to the specified strategy.
        
        Args:
            dataset: Dataset to batch
            seed: Random seed for shuffling
            
        Returns:
            List of batches, where each batch is a list of indices
        """
        if seed is not None:
            torch.manual_seed(seed)
            np.random.seed(seed)
        
        if self.strategy == 'config_only':
            return self._batch_by_config(dataset)
        elif self.strategy == 'length_only':
            return self._batch_by_length(dataset)
        elif self.strategy == 'config_then_length':
            return self._batch_by_config_then_length(dataset)
        else:
            raise ValueError(f"Unknown batching strategy: {self.strategy}")
    
    def _batch_by_config(self, dataset: Dataset) -> List[List[int]]:
        """Batch sequences by hierarchical configuration"""
        print(f"Batching by configuration...")
        
        # Group by (L, m) configuration
        config_groups = defaultdict(list)
        for i in range(len(dataset)):
            config_key = (dataset[i]['config_L'], dataset[i]['config_m'])
            config_groups[config_key].append(i)
        
        # Create batches within each configuration group
        all_batches = []
        for config_key, indices in config_groups.items():
            L, m = config_key
            print(f"  Config L={L}, m={m}: {len(indices)} sequences")
            
            if self.shuffle_within_groups:
                indices = np.random.permutation(indices).tolist()
            
            # Split into batches
            config_batches = [
                indices[i:i + self.max_batch_size]
                for i in range(0, len(indices), self.max_batch_size)
            ]
            all_batches.extend(config_batches)
        
        print(f"  Created {len(all_batches)} batches")
        return all_batches
    
    def _batch_by_length(self, dataset: Dataset) -> List[List[int]]:
        """Batch sequences by length buckets"""
        print(f"Batching by length (bucket size: {self.length_bucket_size})...")
        
        # Group by length buckets
        length_groups = defaultdict(list)
        for i in range(len(dataset)):
            length = dataset[i]['length']
            bucket = (length // self.length_bucket_size) * self.length_bucket_size
            bucket_key = f"{bucket}-{bucket + self.length_bucket_size - 1}"
            length_groups[bucket_key].append(i)
        
        # Create batches within each length group
        all_batches = []
        for bucket_key, indices in length_groups.items():
            print(f"  Length bucket {bucket_key}: {len(indices)} sequences")
            
            if self.shuffle_within_groups:
                indices = np.random.permutation(indices).tolist()
            
            # Split into batches
            length_batches = [
                indices[i:i + self.max_batch_size]
                for i in range(0, len(indices), self.max_batch_size)
            ]
            all_batches.extend(length_batches)
        
        print(f"  Created {len(all_batches)} batches")
        return all_batches
    
    def _batch_by_config_then_length(self, dataset: Dataset) -> List[List[int]]:
        """Batch by configuration first, then by length within each config"""
        print(f"Batching by config then length...")
        
        # Group by (L, m) configuration first
        config_groups = defaultdict(list)
        for i in range(len(dataset)):
            config_key = (dataset[i]['config_L'], dataset[i]['config_m'])
            config_groups[config_key].append(i)
        
        all_batches = []
        for config_key, config_indices in config_groups.items():
            L, m = config_key
            print(f"  Config L={L}, m={m}: {len(config_indices)} sequences")
            
            # Within this config, group by length
            length_groups = defaultdict(list)
            for idx in config_indices:
                length = dataset[idx]['length']
                bucket = (length // self.length_bucket_size) * self.length_bucket_size
                bucket_key = f"{bucket}-{bucket + self.length_bucket_size - 1}"
                length_groups[bucket_key].append(idx)
            
            # Create batches within each length group
            for bucket_key, indices in length_groups.items():
                if self.shuffle_within_groups:
                    indices = np.random.permutation(indices).tolist()
                
                # Split into batches
                config_length_batches = [
                    indices[i:i + self.max_batch_size]
                    for i in range(0, len(indices), self.max_batch_size)
                ]
                all_batches.extend(config_length_batches)
                
                print(f"    Length bucket {bucket_key}: {len(indices)} sequences -> {len(config_length_batches)} batches")
        
        print(f"  Total batches created: {len(all_batches)}")
        return all_batches
    
    def get_batch_statistics(self, dataset: Dataset, batches: List[List[int]]) -> Dict[str, Any]:
        """Get statistics about the created batches"""
        
        batch_sizes = [len(batch) for batch in batches]
        batch_length_variances = []
        
        for batch_indices in batches:
            lengths = [dataset[i]['length'] for i in batch_indices]
            if len(lengths) > 1:
                batch_length_variances.append(np.var(lengths))
            else:
                batch_length_variances.append(0.0)
        
        # Configuration coherence: how many batches contain only one config
        config_coherent_batches = 0
        for batch_indices in batches:
            configs = set((dataset[i]['config_L'], dataset[i]['config_m']) for i in batch_indices)
            if len(configs) == 1:
                config_coherent_batches += 1
        
        return {
            'total_batches': len(batches),
            'avg_batch_size': np.mean(batch_sizes),
            'min_batch_size': min(batch_sizes),
            'max_batch_size': max(batch_sizes),
            'avg_length_variance': np.mean(batch_length_variances),
            'config_coherent_batches': config_coherent_batches,
            'config_coherence_ratio': config_coherent_batches / len(batches) if batches else 0.0
        }


class ConfigAwareSampler(torch.utils.data.Sampler):
    """
    Custom sampler that respects hierarchical configuration grouping.
    """
    
    def __init__(self, 
                 dataset: Dataset, 
                 batcher: DynamicBatcher,
                 shuffle: bool = True,
                 seed: Optional[int] = None):
        """
        Initialize configuration-aware sampler.
        
        Args:
            dataset: Dataset to sample from
            batcher: Dynamic batcher instance
            shuffle: Whether to shuffle batch order
            seed: Random seed
        """
        self.dataset = dataset
        self.batcher = batcher
        self.shuffle = shuffle
        self.seed = seed
        
        # Create batches
        self.batches = self.batcher.create_batches(dataset, seed)
        
        # Get statistics
        self.stats = self.batcher.get_batch_statistics(dataset, self.batches)
        
        print(f"Sampler created with {len(self.batches)} batches")
        print(f"Configuration coherence: {self.stats['config_coherence_ratio']:.2%}")
        print(f"Average length variance: {self.stats['avg_length_variance']:.2f}")
    
    def __iter__(self):
        """Iterate over batch indices"""
        batch_order = list(range(len(self.batches)))
        
        if self.shuffle:
            if self.seed is not None:
                torch.manual_seed(self.seed)
            batch_order = torch.randperm(len(self.batches)).tolist()
        
        for batch_idx in batch_order:
            for sample_idx in self.batches[batch_idx]:
                yield sample_idx
    
    def __len__(self):
        """Return total number of samples"""
        return len(self.dataset)
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get sampler statistics"""
        return self.stats

## Task-specific Dataloader

In [6]:
import torch
from torch.utils.data import DataLoader
from typing import Dict, Any, Optional
from pathlib import Path


class RHMDataLoaderFactory:
    """
    Unified factory for creating task-specific DataLoaders for RHM datasets.
    """
    
    def __init__(self, dataset_path: str, vocab_size: int = 32):
        """
        Initialize the DataLoader factory.
        
        Args:
            dataset_path: Path to the unified RHM dataset
            vocab_size: Vocabulary size of the RHM data
        """
        self.dataset_path = Path(dataset_path)
        self.vocab_size = vocab_size
        
        # Load unified dataset
        print("Loading unified RHM dataset...")
        self.unified_dataset = load_unified_dataset(str(self.dataset_path))
        self.unified_dataset.print_summary()
    
    def create_dataloader(self,
                         task_name: str,
                         batch_size: int = 32,
                         shuffle: bool = True,
                         num_workers: int = 0,
                         # Task-specific parameters
                         max_length: Optional[int] = None,
                         mask_probability: float = 0.15,
                         mask_strategy: str = 'random',
                         # Batching parameters
                         batching_strategy: str = 'config_then_length',
                         length_bucket_size: int = 50,
                         # Filtering parameters
                         filter_config_L: Optional[int] = None,
                         filter_config_m: Optional[int] = None,
                         filter_min_length: Optional[int] = None,
                         filter_max_length: Optional[int] = None,
                         # Other parameters
                         seed: Optional[int] = None) -> Tuple[DataLoader, Dict[str, Any]]:
        """
        Create a task-specific DataLoader.
        
        Args:
            task_name: 'clm' or 'mlm'
            batch_size: Batch size
            shuffle: Whether to shuffle data
            num_workers: Number of workers for DataLoader
            max_length: Maximum sequence length
            mask_probability: Probability of masking tokens (MLM only)
            mask_strategy: Masking strategy (MLM only)
            batching_strategy: Dynamic batching strategy
            length_bucket_size: Size of length buckets
            filter_config_L: Filter by hierarchy depth
            filter_config_m: Filter by multiplicity
            filter_min_length: Filter by minimum length
            filter_max_length: Filter by maximum length
            seed: Random seed
            
        Returns:
            DataLoader and metadata dictionary
        """
        
        print(f"\n{'='*60}")
        print(f"CREATING {task_name.upper()} DATALOADER")
        print(f"{'='*60}")
        
        # Step 1: Filter dataset if needed
        dataset = self.unified_dataset.get_dataset()
        
        if any([filter_config_L, filter_config_m, filter_min_length, filter_max_length]):
            print("Applying filters...")
            
            if filter_config_L is not None or filter_config_m is not None:
                dataset = self.unified_dataset.filter_by_config(L=filter_config_L, m=filter_config_m)
                print(f"  Config filter: {len(dataset)} sequences remaining")
            
            if filter_min_length is not None or filter_max_length is not None:
                dataset = self.unified_dataset.filter_by_length(
                    min_length=filter_min_length, 
                    max_length=filter_max_length
                )
                print(f"  Length filter: {len(dataset)} sequences remaining")
        
        # Step 2: Create task configuration
        task_config = TaskConfig(
            task_name=task_name,
            max_length=max_length,
            mask_probability=mask_probability,
            mask_strategy=mask_strategy,
            pad_token_id=0,
            mask_token_id=self.vocab_size + 1,
            cls_token_id=self.vocab_size + 2,
            sep_token_id=self.vocab_size + 3
        )
        
        print(f"Task configuration:")
        print(f"  Max length: {task_config.max_length}")
        if task_name == 'mlm':
            print(f"  Mask probability: {task_config.mask_probability}")
            print(f"  Mask strategy: {task_config.mask_strategy}")
        
        # Step 3: Process dataset for specific task
        processor = ProcessorFactory.create_processor(task_name, task_config, self.vocab_size)
        processed_dataset = processor.process_dataset(dataset)
        
        # Step 4: Create dynamic batcher
        batcher = DynamicBatcher(
            strategy=batching_strategy,
            length_bucket_size=length_bucket_size,
            max_batch_size=batch_size,
            shuffle_within_groups=shuffle
        )
        
        print(f"Batching strategy: {batching_strategy}")
        
        # Step 5: Create custom sampler if using dynamic batching
        if batching_strategy != 'none':
            sampler = ConfigAwareSampler(
                dataset=processed_dataset,
                batcher=batcher,
                shuffle=shuffle,
                seed=seed
            )
            shuffle = False  # Disable DataLoader shuffle when using custom sampler
        else:
            sampler = None
        
        # Step 6: Create DataLoader
        dataloader = DataLoader(
            processed_dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            sampler=sampler,
            num_workers=num_workers,
            collate_fn=processor.get_collate_fn(),
            pin_memory=torch.cuda.is_available()
        )
        
        # Step 7: Create metadata
        metadata = {
            'task_name': task_name,
            'task_config': task_config,
            'original_dataset_size': len(self.unified_dataset.get_dataset()),
            'filtered_dataset_size': len(dataset),
            'processed_dataset_size': len(processed_dataset),
            'batch_size': batch_size,
            'batching_strategy': batching_strategy,
            'vocab_info': self.unified_dataset.get_vocab_info(),
            'sampler_stats': sampler.get_statistics() if sampler else None,
            'filters_applied': {
                'config_L': filter_config_L,
                'config_m': filter_config_m,
                'min_length': filter_min_length,
                'max_length': filter_max_length
            }
        }
        
        print(f"\nDataLoader created successfully!")
        print(f"  Original dataset: {metadata['original_dataset_size']:,} sequences")
        print(f"  After filtering: {metadata['filtered_dataset_size']:,} sequences")
        print(f"  After processing: {metadata['processed_dataset_size']:,} sequences")
        print(f"  Batches per epoch: {len(dataloader):,}")
        
        if sampler:
            stats = sampler.get_statistics()
            print(f"  Config coherence: {stats['config_coherence_ratio']:.1%}")
            print(f"  Avg length variance: {stats['avg_length_variance']:.2f}")
        
        print(f"{'='*60}")
        
        return dataloader, metadata

In [7]:
def load_unified_dataset(dataset_path: str) -> UnifiedRHMDataset:
    """
    Convenience function to load a unified RHM dataset.
    
    Args:
        dataset_path: Path to directory containing raw RHM dataset
        
    Returns:
        UnifiedRHMDataset instance
    """
    return UnifiedRHMDataset(dataset_path)

## main function to test step 2-5

In [9]:
def main():
    """Main script demonstrating the unified pipeline"""
    
    # Configuration
    dataset_path = './raw_rhm_data'
    vocab_size = 32
    
    try:
        # Initialize the factory
        factory = RHMDataLoaderFactory(dataset_path, vocab_size)
        
        # Example 1: Create CLM DataLoader
        print("\n" + "="*80)
        print("EXAMPLE 1: CAUSAL LANGUAGE MODELING")
        print("="*80)
        
        clm_dataloader, clm_metadata = factory.create_dataloader(
            task_name='clm',
            batch_size=16,
            max_length=1024,
            batching_strategy='config_then_length',
            length_bucket_size=100,
            seed=42
        )
        
        # Test CLM batch
        print("\nTesting CLM DataLoader...")
        for i, batch in enumerate(clm_dataloader):
            print(f"CLM Batch {i+1}:")
            print(f"  Input IDs shape: {batch['input_ids'].shape}")
            print(f"  Labels shape: {batch['labels'].shape}")
            print(f"  Attention mask shape: {batch['attention_mask'].shape}")
            print(f"  Configs in batch: {set(zip(batch['config_L'], batch['config_m']))}")
            print(f"  Sample input (first 10 tokens): {batch['input_ids'][0][:10].tolist()}")
            print(f"  Sample labels (first 10 tokens): {batch['labels'][0][:10].tolist()}")
            if i >= 2:  # Show only first 3 batches
                break
        
        # Example 2: Create MLM DataLoader  
        print("\n" + "="*80)
        print("EXAMPLE 2: MASKED LANGUAGE MODELING")
        print("="*80)
        
        mlm_dataloader, mlm_metadata = factory.create_dataloader(
            task_name='mlm',
            batch_size=16,
            max_length=512,
            mask_probability=0.15,
            mask_strategy='random',
            batching_strategy='config_then_length',
            length_bucket_size=50,
            seed=42
        )
        
        # Test MLM batch
        print("\nTesting MLM DataLoader...")
        for i, batch in enumerate(mlm_dataloader):
            print(f"MLM Batch {i+1}:")
            print(f"  Input IDs shape: {batch['input_ids'].shape}")
            print(f"  Labels shape: {batch['labels'].shape}")
            print(f"  Attention mask shape: {batch['attention_mask'].shape}")
            print(f"  Configs in batch: {set(zip(batch['config_L'], batch['config_m']))}")
            
            # Show masking example
            input_ids = batch['input_ids'][0]
            labels = batch['labels'][0]
            mask_token_id = vocab_size + 1
            
            masked_positions = (input_ids == mask_token_id).nonzero().flatten()
            if len(masked_positions) > 0:
                print(f"  Masked positions: {masked_positions[:5].tolist()}")  # Show first 5
                print(f"  Original tokens at masked positions: {labels[masked_positions[:5]].tolist()}")
            
            if i >= 2:  # Show only first 3 batches
                break
        
        # Example 3: Configuration-specific DataLoader
        print("\n" + "="*80)
        print("EXAMPLE 3: CONFIGURATION-SPECIFIC LOADING")
        print("="*80)
        
        config_dataloader, config_metadata = factory.create_dataloader(
            task_name='mlm',
            batch_size=8,
            max_length=512,
            mask_strategy='hierarchical',
            batching_strategy='config_only',
            filter_config_L=2,  # Only L=2 configurations
            filter_config_m=2,  # Only m=2 configurations
            seed=42
        )
        
        print(f"\nConfig-specific dataset:")
        print(f"  Filtered to L=2, m=2 only")
        print(f"  Total batches: {len(config_dataloader)}")
        
        # Test config-specific batch
        for i, batch in enumerate(config_dataloader):
            print(f"Config Batch {i+1}:")
            print(f"  All configs should be (2, 2): {set(zip(batch['config_L'], batch['config_m']))}")
            if i >= 1:  # Show only first 2 batches
                break
        
        # Example 4: Different batching strategies comparison
        print("\n" + "="*80)
        print("EXAMPLE 4: BATCHING STRATEGY COMPARISON")
        print("="*80)
        
        strategies = ['config_only', 'length_only', 'config_then_length']
        
        for strategy in strategies:
            print(f"\nTesting strategy: {strategy}")
            
            test_dataloader, test_metadata = factory.create_dataloader(
                task_name='clm',
                batch_size=16,
                batching_strategy=strategy,
                length_bucket_size=100,
                seed=42
            )
            
            # Analyze first few batches
            config_coherence_count = 0
            length_variances = []
            
            for i, batch in enumerate(test_dataloader):
                # Check config coherence
                configs = set(zip(batch['config_L'], batch['config_m']))
                if len(configs) == 1:
                    config_coherence_count += 1
                
                # Check length variance
                lengths = [len([x for x in seq if x != 0]) for seq in batch['input_ids']]  # Exclude padding
                if len(lengths) > 1:
                    length_variances.append(torch.var(torch.tensor(lengths, dtype=torch.float)).item())
                
                if i >= 19:  # Analyze first 20 batches
                    break
            
            coherence_ratio = config_coherence_count / min(20, len(test_dataloader))
            avg_length_variance = sum(length_variances) / len(length_variances) if length_variances else 0
            
            print(f"  Config coherence: {coherence_ratio:.1%}")
            print(f"  Avg length variance: {avg_length_variance:.2f}")
        
        # Example 5: Export metadata for analysis
        print("\n" + "="*80)
        print("EXAMPLE 5: METADATA EXPORT")
        print("="*80)
        
        # Save metadata for further analysis
        import json
        
        analysis_data = {
            'clm_metadata': clm_metadata,
            'mlm_metadata': mlm_metadata,
            'config_metadata': config_metadata,
            'dataset_statistics': factory.unified_dataset.get_statistics(),
            'vocab_info': factory.unified_dataset.get_vocab_info()
        }
        
        # Convert non-serializable objects to strings
        def make_serializable(obj):
            if hasattr(obj, '__dict__'):
                return str(obj)
            elif isinstance(obj, torch.Tensor):
                return obj.tolist()
            return obj
        
        # Clean metadata for JSON serialization
        clean_analysis = {}
        for key, value in analysis_data.items():
            if isinstance(value, dict):
                clean_analysis[key] = {k: make_serializable(v) for k, v in value.items()}
            else:
                clean_analysis[key] = make_serializable(value)
        
        # Save analysis
        output_path = Path('./dataloader_analysis.json')
        with output_path.open('w') as f:
            json.dump(clean_analysis, f, indent=2, default=str)
        
        print(f"✓ Analysis saved to {output_path}")
        
        print("\n" + "="*80)
        print("PIPELINE DEMONSTRATION COMPLETE")
        print("="*80)
        print("\nSummary:")
        print(f"✓ Successfully created CLM DataLoader with {len(clm_dataloader)} batches")
        print(f"✓ Successfully created MLM DataLoader with {len(mlm_dataloader)} batches")
        print(f"✓ Demonstrated configuration-specific filtering")
        print(f"✓ Compared different batching strategies")
        print(f"✓ Exported metadata for analysis")
        print("\nThe unified pipeline is ready for training!")
        
    except FileNotFoundError as e:
        print(f"❌ Error: {e}")
        print("\n💡 Solution:")
        print("1. Run the raw dataset generation script first:")
        print("   python generate_raw_dataset.py")
        print("2. Ensure the dataset path is correct")
        
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

Loading unified RHM dataset...
Loading RHM dataset...
✓ Loaded dataset with 5000 sequences
✓ Loaded metadata for 5 configurations
Validating dataset structure...
✓ Dataset validation passed
Computing dataset statistics...
✓ Statistics computed

UNIFIED RHM DATASET SUMMARY
Total sequences: 5,000
Total tokens: 40,000
Sequence length: 4-16 (avg: 8.0)
Vocabulary: 1-32 (0 reserved)

Configurations (5):
----------------------------------------
Task 0: L=2, m=2
  Sequences: 1,000 (20.0%)
  Length: 4-4 (avg: 4.0)
  Tokens: 4,000
Task 1: L=3, m=2
  Sequences: 1,000 (20.0%)
  Length: 8-8 (avg: 8.0)
  Tokens: 8,000
Task 2: L=2, m=4
  Sequences: 1,000 (20.0%)
  Length: 4-4 (avg: 4.0)
  Tokens: 4,000
Task 3: L=4, m=2
  Sequences: 1,000 (20.0%)
  Length: 16-16 (avg: 16.0)
  Tokens: 16,000
Task 4: L=3, m=3
  Sequences: 1,000 (20.0%)
  Length: 8-8 (avg: 8.0)
  Tokens: 8,000

EXAMPLE 1: CAUSAL LANGUAGE MODELING

CREATING CLM DATALOADER
Task configuration:
  Max length: 1024
Processing dataset for Causa

# Step 4: Training

In [8]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR
from transformers import (
    AutoConfig, AutoModelForCausalLM, AutoModelForMaskedLM,
    get_linear_schedule_with_warmup, TrainingArguments, Trainer,
    EarlyStoppingCallback, TrainerCallback
)
from transformers.modeling_outputs import CausalLMOutput, MaskedLMOutput
import numpy as np
from pathlib import Path
from typing import Dict, Any, Optional, List, Union, Tuple
import json
import time
from datetime import datetime
import wandb
from dataclasses import dataclass, field
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class RHMTrainingConfig:
    """Configuration for RHM training"""
    
    # Model configuration
    model_name_or_path: Optional[str] = None  # For loading pretrained models
    vocab_size: int = 37  # 32 + 4 special tokens + 1 for safety
    hidden_size: int = 512
    num_hidden_layers: int = 6
    num_attention_heads: int = 8
    intermediate_size: int = 2048
    max_position_embeddings: int = 2048
    
    # Training configuration
    task_name: str = 'clm'  # 'clm' or 'mlm'
    output_dir: str = './rhm_training_output'
    num_train_epochs: int = 10
    per_device_train_batch_size: int = 16
    per_device_eval_batch_size: int = 32
    gradient_accumulation_steps: int = 1
    learning_rate: float = 5e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    lr_scheduler_type: str = 'linear'
    
    # Checkpoint configuration
    save_strategy: str = 'steps'  # 'steps', 'epoch', 'no'
    save_steps: int = 500
    save_total_limit: int = 5
    load_best_model_at_end: bool = True
    metric_for_best_model: str = 'eval_loss'
    greater_is_better: bool = False
    
    # Evaluation configuration
    evaluation_strategy: str = 'steps'  # 'steps', 'epoch', 'no'
    eval_steps: int = 500
    eval_accumulation_steps: Optional[int] = None
    
    # Logging configuration
    logging_strategy: str = 'steps'
    logging_steps: int = 100
    report_to: List[str] = field(default_factory=lambda: ['tensorboard'])
    run_name: Optional[str] = None
    
    # Optimization configuration
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    adam_epsilon: float = 1e-8
    max_grad_norm: float = 1.0
    
    # Early stopping
    early_stopping: bool = True
    early_stopping_patience: int = 3
    early_stopping_threshold: float = 0.0
    
    # Mixed precision
    fp16: bool = False
    bf16: bool = False
    
    # Data configuration
    dataloader_num_workers: int = 0
    dataloader_pin_memory: bool = True
    remove_unused_columns: bool = False
    
    # Hierarchical analysis
    track_hierarchical_metrics: bool = True
    hierarchical_eval_frequency: int = 1000
    
    # Reproducibility
    seed: int = 42
    
    def to_training_arguments(self) -> TrainingArguments:
        """Convert to HuggingFace TrainingArguments"""
        return TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=self.num_train_epochs,
            per_device_train_batch_size=self.per_device_train_batch_size,
            per_device_eval_batch_size=self.per_device_eval_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            weight_decay=self.weight_decay,
            warmup_ratio=self.warmup_ratio,
            lr_scheduler_type=self.lr_scheduler_type,
            save_strategy=self.save_strategy,
            save_steps=self.save_steps,
            save_total_limit=self.save_total_limit,
            load_best_model_at_end=self.load_best_model_at_end,
            metric_for_best_model=self.metric_for_best_model,
            greater_is_better=self.greater_is_better,
            evaluation_strategy=self.evaluation_strategy,
            eval_steps=self.eval_steps,
            eval_accumulation_steps=self.eval_accumulation_steps,
            logging_strategy=self.logging_strategy,
            logging_steps=self.logging_steps,
            report_to=self.report_to,
            run_name=self.run_name,
            adam_beta1=self.adam_beta1,
            adam_beta2=self.adam_beta2,
            adam_epsilon=self.adam_epsilon,
            max_grad_norm=self.max_grad_norm,
            fp16=self.fp16,
            bf16=self.bf16,
            dataloader_num_workers=self.dataloader_num_workers,
            dataloader_pin_memory=self.dataloader_pin_memory,
            remove_unused_columns=self.remove_unused_columns,
            seed=self.seed,
        )


class HierarchicalMetricsCallback(TrainerCallback):
    """Callback to track hierarchical-specific metrics"""
    
    def __init__(self, config: RHMTrainingConfig, dataloader_metadata: Dict[str, Any]):
        self.config = config
        self.dataloader_metadata = dataloader_metadata
        self.hierarchical_metrics = []
    
    def on_evaluate(self, args, state, control, model, eval_dataloader, **kwargs):
        """Called after evaluation"""
        if not self.config.track_hierarchical_metrics:
            return
        
        if state.global_step % self.config.hierarchical_eval_frequency == 0:
            metrics = self._compute_hierarchical_metrics(model, eval_dataloader)
            self.hierarchical_metrics.append({
                'step': state.global_step,
                'epoch': state.epoch,
                'metrics': metrics
            })
            
            # Log to wandb if available
            if 'wandb' in self.config.report_to:
                wandb.log({f"hierarchical/{k}": v for k, v in metrics.items()}, 
                         step=state.global_step)
    
    def _compute_hierarchical_metrics(self, model, dataloader) -> Dict[str, float]:
        """Compute metrics specific to hierarchical configurations"""
        model.eval()
        
        config_losses = {}
        config_counts = {}
        
        with torch.no_grad():
            for batch in dataloader:
                # Get predictions
                if hasattr(batch, 'to'):
                    batch = batch.to(model.device)
                
                outputs = model(**{k: v for k, v in batch.items() 
                               if k in ['input_ids', 'attention_mask', 'labels']})
                
                losses = outputs.loss
                
                # Group by configuration
                for i, (L, m) in enumerate(zip(batch['config_L'], batch['config_m'])):
                    config_key = f"L{L}_m{m}"
                    
                    if config_key not in config_losses:
                        config_losses[config_key] = 0.0
                        config_counts[config_key] = 0
                    
                    # Individual sample loss (approximation)
                    if len(losses.shape) == 0:  # Scalar loss
                        sample_loss = losses.item()
                    else:
                        sample_loss = losses[i].item() if len(losses) > i else losses.mean().item()
                    
                    config_losses[config_key] += sample_loss
                    config_counts[config_key] += 1
        
        # Compute average losses per configuration
        hierarchical_metrics = {}
        for config_key in config_losses:
            avg_loss = config_losses[config_key] / config_counts[config_key]
            hierarchical_metrics[f"loss_{config_key}"] = avg_loss
        
        model.train()
        return hierarchical_metrics


class CheckpointCallback(TrainerCallback):
    """Enhanced checkpoint callback with hierarchical metadata"""
    
    def __init__(self, config: RHMTrainingConfig, dataloader_metadata: Dict[str, Any]):
        self.config = config
        self.dataloader_metadata = dataloader_metadata
        self.checkpoint_history = []
    
    def on_save(self, args, state, control, model, tokenizer, **kwargs):
        """Called when saving checkpoint"""
        checkpoint_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        
        # Save enhanced metadata
        enhanced_metadata = {
            'training_config': self.config.__dict__,
            'dataloader_metadata': self.dataloader_metadata,
            'training_state': {
                'global_step': state.global_step,
                'epoch': state.epoch,
                'learning_rate': state.log_history[-1].get('learning_rate', 0) if state.log_history else 0,
                'train_loss': state.log_history[-1].get('train_loss', 0) if state.log_history else 0,
                'eval_loss': state.log_history[-1].get('eval_loss', 0) if state.log_history else 0,
            },
            'model_config': model.config.to_dict() if hasattr(model.config, 'to_dict') else str(model.config),
            'timestamp': datetime.now().isoformat(),
        }
        
        # Save metadata
        metadata_path = checkpoint_dir / 'training_metadata.json'
        with metadata_path.open('w') as f:
            json.dump(enhanced_metadata, f, indent=2, default=str)
        
        # Track checkpoint
        self.checkpoint_history.append({
            'step': state.global_step,
            'epoch': state.epoch,
            'path': str(checkpoint_dir),
            'timestamp': datetime.now().isoformat()
        })
        
        logger.info(f"Enhanced checkpoint saved to {checkpoint_dir}")


class RHMTrainer:
    """Main trainer class for RHM models"""
    
    def __init__(self, 
                 training_config: RHMTrainingConfig,
                 train_dataloader,
                 eval_dataloader, 
                 dataloader_metadata: Dict[str, Any]):
        """
        Initialize RHM trainer.
        
        Args:
            training_config: Training configuration
            train_dataloader: Training DataLoader
            eval_dataloader: Evaluation DataLoader
            dataloader_metadata: Metadata from DataLoader creation
        """
        self.config = training_config
        self.train_dataloader = train_dataloader
        self.eval_dataloader = eval_dataloader
        self.dataloader_metadata = dataloader_metadata
        
        # Set up output directory
        self.output_dir = Path(self.config.output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Initialize model
        self.model = self._create_model()
        
        # Set up training arguments
        self.training_args = self.config.to_training_arguments()
        
        # Set up callbacks
        self.callbacks = self._setup_callbacks()
        
        # Initialize trainer
        self.trainer = None
        
        logger.info(f"RHM Trainer initialized for task: {self.config.task_name}")
        logger.info(f"Model parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        logger.info(f"Training on device: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")
    
    def _create_model(self):
        """Create model based on task and configuration"""
        
        # Create model configuration
        if self.config.model_name_or_path:
            # Load from existing model
            model_config = AutoConfig.from_pretrained(self.config.model_name_or_path)
            model_config.vocab_size = self.config.vocab_size
        else:
            # Create new configuration
            if self.config.task_name == 'clm':
                from transformers import GPT2Config
                model_config = GPT2Config(
                    vocab_size=self.config.vocab_size,
                    n_positions=self.config.max_position_embeddings,
                    n_embd=self.config.hidden_size,
                    n_layer=self.config.num_hidden_layers,
                    n_head=self.config.num_attention_heads,
                    n_inner=self.config.intermediate_size,
                    resid_pdrop=0.1,
                    embd_pdrop=0.1,
                    attn_pdrop=0.1,
                    use_cache=False,  # Disable for training
                )
            elif self.config.task_name == 'mlm':
                from transformers import BertConfig
                model_config = BertConfig(
                    vocab_size=self.config.vocab_size,
                    hidden_size=self.config.hidden_size,
                    num_hidden_layers=self.config.num_hidden_layers,
                    num_attention_heads=self.config.num_attention_heads,
                    intermediate_size=self.config.intermediate_size,
                    max_position_embeddings=self.config.max_position_embeddings,
                    hidden_dropout_prob=0.1,
                    attention_probs_dropout_prob=0.1,
                )
            else:
                raise ValueError(f"Unknown task: {self.config.task_name}")
        
        # Create model
        if self.config.model_name_or_path:
            if self.config.task_name == 'clm':
                model = AutoModelForCausalLM.from_pretrained(
                    self.config.model_name_or_path,
                    config=model_config
                )
            else:
                model = AutoModelForMaskedLM.from_pretrained(
                    self.config.model_name_or_path,
                    config=model_config
                )
        else:
            if self.config.task_name == 'clm':
                model = AutoModelForCausalLM.from_config(model_config)
            else:
                model = AutoModelForMaskedLM.from_config(model_config)
        
        # Move to device
        if torch.cuda.is_available():
            model = model.cuda()
        
        return model
    
    def _setup_callbacks(self) -> List[TrainerCallback]:
        """Set up training callbacks"""
        callbacks = []
        
        # Hierarchical metrics callback
        if self.config.track_hierarchical_metrics:
            callbacks.append(HierarchicalMetricsCallback(self.config, self.dataloader_metadata))
        
        # Enhanced checkpoint callback
        callbacks.append(CheckpointCallback(self.config, self.dataloader_metadata))
        
        # Early stopping callback
        if self.config.early_stopping:
            callbacks.append(EarlyStoppingCallback(
                early_stopping_patience=self.config.early_stopping_patience,
                early_stopping_threshold=self.config.early_stopping_threshold
            ))
        
        return callbacks
    
    def train(self) -> Dict[str, Any]:
        """Run training"""
        
        logger.info("Starting training...")
        logger.info(f"Task: {self.config.task_name}")
        logger.info(f"Training samples: {len(self.train_dataloader.dataset)}")
        logger.info(f"Evaluation samples: {len(self.eval_dataloader.dataset)}")
        logger.info(f"Epochs: {self.config.num_train_epochs}")
        logger.info(f"Batch size: {self.config.per_device_train_batch_size}")
        logger.info(f"Learning rate: {self.config.learning_rate}")
        
        # Save initial configuration
        self._save_training_setup()
        
        # Initialize trainer
        self.trainer = Trainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.train_dataloader.dataset,
            eval_dataset=self.eval_dataloader.dataset,
            data_collator=self.train_dataloader.collate_fn,
            callbacks=self.callbacks,
        )
        
        # Start training
        start_time = time.time()
        
        try:
            train_result = self.trainer.train()
            training_time = time.time() - start_time
            
            # Save final model
            self.trainer.save_model()
            
            # Collect final metrics
            final_metrics = {
                'train_result': train_result.metrics,
                'training_time': training_time,
                'final_eval_metrics': self.trainer.evaluate(),
                'model_size': sum(p.numel() for p in self.model.parameters()),
                'dataset_info': self.dataloader_metadata,
            }
            
            # Save final metrics
            metrics_path = self.output_dir / 'final_metrics.json'
            with metrics_path.open('w') as f:
                json.dump(final_metrics, f, indent=2, default=str)
            
            logger.info(f"Training completed in {training_time:.2f} seconds")
            logger.info(f"Final evaluation loss: {final_metrics['final_eval_metrics']['eval_loss']:.4f}")
            
            return final_metrics
            
        except Exception as e:
            logger.error(f"Training failed: {e}")
            # Save error information
            error_info = {
                'error': str(e),
                'timestamp': datetime.now().isoformat(),
                'training_config': self.config.__dict__,
                'dataloader_metadata': self.dataloader_metadata
            }
            
            error_path = self.output_dir / 'training_error.json'
            with error_path.open('w') as f:
                json.dump(error_info, f, indent=2, default=str)
            
            raise
    
    def _save_training_setup(self):
        """Save complete training setup for reproducibility"""
        
        setup_info = {
            'training_config': self.config.__dict__,
            'dataloader_metadata': self.dataloader_metadata,
            'model_config': self.model.config.to_dict() if hasattr(self.model.config, 'to_dict') else str(self.model.config),
            'training_arguments': self.training_args.to_dict(),
            'device_info': {
                'cuda_available': torch.cuda.is_available(),
                'cuda_device_count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
                'cuda_device_name': torch.cuda.get_device_name() if torch.cuda.is_available() else None,
            },
            'pytorch_version': torch.__version__,
            'timestamp': datetime.now().isoformat(),
        }
        
        setup_path = self.output_dir / 'training_setup.json'
        with setup_path.open('w') as f:
            json.dump(setup_info, f, indent=2, default=str)
        
        logger.info(f"Training setup saved to {setup_path}")



In [9]:
def create_trainer(training_config: RHMTrainingConfig,
                  train_dataloader,
                  eval_dataloader,
                  dataloader_metadata: Dict[str, Any]) -> RHMTrainer:
    """
    Convenience function to create an RHM trainer.
    
    Args:
        training_config: Training configuration
        train_dataloader: Training DataLoader
        eval_dataloader: Evaluation DataLoader  
        dataloader_metadata: Metadata from DataLoader creation
        
    Returns:
        RHMTrainer instance
    """
    return RHMTrainer(training_config, train_dataloader, eval_dataloader, dataloader_metadata)


def main_training_example():
    """Example of how to use the training framework"""
    
    
    # Initialize factory and create DataLoaders
    factory = RHMDataLoaderFactory('./raw_rhm_data', vocab_size=32)
    
    # Create train DataLoader
    train_dataloader, train_metadata = factory.create_dataloader(
        task_name='clm',
        batch_size=16,
        max_length=1024,
        batching_strategy='config_then_length',
        seed=42
    )
    
    # Create eval DataLoader (subset for faster evaluation)
    eval_dataloader, eval_metadata = factory.create_dataloader(
        task_name='clm',
        batch_size=32,
        max_length=1024,
        batching_strategy='config_then_length',
        filter_max_length=512,  # Smaller sequences for eval
        seed=42
    )
    
# Create training configuration
    training_config = RHMTrainingConfig(
        task_name='clm',
        output_dir='./rhm_clm_training',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=5e-4,
        warmup_ratio=0.1,
        save_steps=500,
        eval_steps=500,
        logging_steps=100,
        save_total_limit=3,
        load_best_model_at_end=True,
        track_hierarchical_metrics=True,
        early_stopping=True,
        early_stopping_patience=3,
        run_name=f"rhm_clm_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
        seed=42
    )
    
    # Create and run trainer
    trainer = create_trainer(
        training_config=training_config,
        train_dataloader=train_dataloader,
        eval_dataloader=eval_dataloader,
        dataloader_metadata=train_metadata
    )
    
    # Train model
    results = trainer.train()
    
    print(f"Training completed! Results saved to {training_config.output_dir}")
    return results


In [11]:
if __name__ == "__main__":
    main_training_example()

Loading unified RHM dataset...
Loading RHM dataset...
✓ Loaded dataset with 5000 sequences
✓ Loaded metadata for 5 configurations
Validating dataset structure...
✓ Dataset validation passed
Computing dataset statistics...
✓ Statistics computed

UNIFIED RHM DATASET SUMMARY
Total sequences: 5,000
Total tokens: 40,000
Sequence length: 4-16 (avg: 8.0)
Vocabulary: 1-32 (0 reserved)

Configurations (5):
----------------------------------------
Task 0: L=2, m=2
  Sequences: 1,000 (20.0%)
  Length: 4-4 (avg: 4.0)
  Tokens: 4,000
Task 1: L=3, m=2
  Sequences: 1,000 (20.0%)
  Length: 8-8 (avg: 8.0)
  Tokens: 8,000
Task 2: L=2, m=4
  Sequences: 1,000 (20.0%)
  Length: 4-4 (avg: 4.0)
  Tokens: 4,000
Task 3: L=4, m=2
  Sequences: 1,000 (20.0%)
  Length: 16-16 (avg: 16.0)
  Tokens: 16,000
Task 4: L=3, m=3
  Sequences: 1,000 (20.0%)
  Length: 8-8 (avg: 8.0)
  Tokens: 8,000

CREATING CLM DATALOADER
Task configuration:
  Max length: 1024
Processing dataset for Causal Language Modeling...
Original seque

INFO:__main__:RHM Trainer initialized for task: clm
INFO:__main__:Model parameters: 19,982,848
INFO:__main__:Training on device: CPU
INFO:__main__:Starting training...
INFO:__main__:Task: clm
INFO:__main__:Training samples: 5000
INFO:__main__:Evaluation samples: 5000
INFO:__main__:Epochs: 5
INFO:__main__:Batch size: 16
INFO:__main__:Learning rate: 0.0005
INFO:__main__:Training setup saved to rhm_clm_training/training_setup.json


RuntimeError: TensorBoardCallback requires tensorboard to be installed. Either update your PyTorch version or install tensorboardX.